# Preparing the text for topic modelling

This notebook turns the plain text produced by `3_tei_to_text.xsl` into the list of sentences that BERTopic works with, and checks the result.

In the input file `lesefassung.txt`, `///` marks the start of a new paragraph and `###` the start of a new sentence.

Steps:
1. Split the text into sentences and record the paragraph number of each sentence.
2. Count the sentences per paragraph.
3. Compare these counts with the number of sentence milestones per paragraph in the TEI file (from `validation/count_milestones_per_paragraph.xsl`). If both agree, the plain text and the TEI file have the same sentence boundaries.

## 1. Split the text into sentences

The text is split at `///` (paragraphs) and `###` (sentences). Spaces are normalised (non-breaking spaces and line breaks become normal spaces). Every sentence is written on its own line together with its paragraph number (`Paragraph 12: …`) to `sentences_for_bert_schwerin_2026_final.txt`.

In [56]:
import re

file_path = '../../02_bertopic/input/lesefassung.txt'
output_file_path = '../output/sentences_for_bert_schwerin_2026_final.txt'

with open(file_path, 'r', encoding='utf-8') as file:
    text = file.read()

def clean_text(sentence):
    # Replace non-breaking spaces and other non-standard spaces
    sentence = sentence.replace('\xa0', ' ').replace('\n', ' ')
    # Replace multiple spaces with a single space
    sentence = re.sub(r'\s+', ' ', sentence).strip()
    return sentence

def process_text_for_bertopic(text):
    # Split the text into paragraphs using '///' as the delimiter
    paragraphs = text.split('///')
    sentences = []
    sentence_paragraph_map = []

    # Enumerate the paragraphs and split sentences at '###'
    for idx, paragraph in enumerate(paragraphs):
        # Trim whitespace which might be left from splitting
        trimmed_paragraph = paragraph.strip()
        if trimmed_paragraph:  # Ensure the paragraph is not empty
            # Split the paragraph into sentences using '###'
            paragraph_sentences = [s.strip() for s in trimmed_paragraph.split('###') if s.strip()]
            # Clean and extend the sentences to the main list
            cleaned_sentences = [clean_text(sentence) for sentence in paragraph_sentences]
            sentences.extend(cleaned_sentences)
            # Store the paragraph number for each sentence for later analysis
            sentence_paragraph_map.extend([idx + 1] * len(cleaned_sentences))

    return sentences, sentence_paragraph_map

# Process the text for BERTopic
sentences, sentence_paragraph_map = process_text_for_bertopic(text)

# Write sentences and paragraph numbers to an output file
with open(output_file_path, 'w', encoding='utf-8') as output_file:
    output_file.write(f"The text contains {len(sentences)} sentences based on the '###' split.\n\n")
    for sentence, paragraph_num in zip(sentences, sentence_paragraph_map):
        output_file.write(f"Paragraph {paragraph_num}: {sentence}\n")

# Print the first 100 sentences to the console
print("The first 100 sentences:")
for i, (sentence, paragraph_num) in enumerate(zip(sentences, sentence_paragraph_map)):
    if i >= 400:  # Stop after printing 100 sentences
        break
    print(f"Paragraph {paragraph_num}: {sentence}")

print(f"All sentences have been written to {output_file_path}.")


The first 100 sentences:
Paragraph 1: Fürst Khevenhüller 1896
Paragraph 2: « Histoire De la Vie de madame la comtesse de Schwerin
Paragraph 3: écrite par elle-même à ses enfants suivant les ordres de son directeur à Cologne »
Paragraph 4: Première partie
Paragraph 5: L’histoire de ma vie est si remarquable et si remplie d’événements, tant pour leur singularité que par le bruit que les plus considérables ont fait dans le monde, que ma curiosité m’a portée en premier lieu d’en rassembler le cours, et secondement pour qu’un jour mes chers enfants puissent être instruits par moi-même des suites malheureuses (selon le monde) que produit ordinairement une fortune éblouissante à une jeune personne élevée dans tout ce qui peut plaire à son ambition et à ses sens, suites dis-je d’autant plus dangereuses qu’elles sont imperceptibles à ceux dont l’amour-propre ferme les yeux sur leur conduite, et dont ils ne sont détrompés que par une longue expérience, d’autant plus rude à être exercée que peu d

## 2. Count the sentences per paragraph

Reads the sentence file written in step 1 and counts how many sentences each paragraph has.

In [57]:
from collections import defaultdict
import re
import pandas as pd

def count_sentences_per_paragraph(text):
    paragraph_counts = defaultdict(int)

    # Find all paragraph labels in the form "Paragraph X"
    matches = re.findall(r"(Paragraph \d+):", text)
    
    # Count occurrences of each paragraph
    for match in matches:
        paragraph_counts[match] += 1

    return paragraph_counts

# Read text from the file 'processed_sentences.txt'
with open('../output/sentences_for_bert_schwerin_2026_final.txt', 'r', encoding='utf-8') as file:
    text = file.read()

# Count sentences per paragraph
sentence_counts = count_sentences_per_paragraph(text)

# Convert the result to a pandas DataFrame for easier manipulation or display
df_sentence_counts = pd.DataFrame(list(sentence_counts.items()), columns=["Paragraph", "Sentence Count"])

# Output the result
print(df_sentence_counts)

# Optionally, you can save the result to a CSV file
df_sentence_counts.to_csv('sentence_counts.csv', index=False)


         Paragraph  Sentence Count
0      Paragraph 1               1
1      Paragraph 2               1
2      Paragraph 3               1
3      Paragraph 4               1
4      Paragraph 5               6
..             ...             ...
673  Paragraph 674              19
674  Paragraph 675              22
675  Paragraph 676              18
676  Paragraph 677               3
677  Paragraph 678               1

[678 rows x 2 columns]


## 3. Compare with the TEI file

Compares the sentence counts from step 2 (`sentence_counts.csv`, written by step 2 into this folder) with the number of `<milestone unit="sentence"/>` per paragraph in the TEI file. The milestone counts come from `counted_milestones.xml`, which is produced by applying `validation/count_milestones_per_paragraph.xsl` to `../output/lesefassung_milestones2.xml` and saving the result in this folder. Paragraphs whose counts differ are listed; for the published data, all paragraphs agree.

In [58]:
import pandas as pd
import xml.etree.ElementTree as ET

# Step 1: Parse the XML file
def parse_xml(file_path):
    tree = ET.parse(file_path)
    root = tree.getroot()
    
    milestone_counts = {}
    # Loop over each paragraph in the XML and extract the ID and milestone count
    for paragraph in root.findall(".//paragraph"):
        paragraph_id = paragraph.attrib['id']
        milestone_count = int(paragraph.find('milestoneCount').text)
        milestone_counts[paragraph_id] = milestone_count
    
    return milestone_counts

# Step 2: Read the CSV file
def parse_csv(file_path):
    df = pd.read_csv(file_path)
    sentence_counts = {}
    
    # Clean up 'Paragraph X' format to match 'P.X' in XML file
    for _, row in df.iterrows():
        paragraph_id = row['Paragraph'].replace('Paragraph ', 'P.')
        sentence_counts[paragraph_id] = row['Sentence Count']
    
    return sentence_counts

# Step 3: Compare the sentence counts
def compare_counts(xml_counts, csv_counts):
    mismatched_paragraphs = []
    
    # Compare paragraph counts between XML and CSV
    for paragraph_id, xml_count in xml_counts.items():
        csv_count = csv_counts.get(paragraph_id)
        
        # If the counts are not equal, record the paragraph
        if csv_count is None:
            mismatched_paragraphs.append((paragraph_id, xml_count, 'Missing in CSV'))
        elif xml_count != csv_count:
            mismatched_paragraphs.append((paragraph_id, xml_count, csv_count))
    
    return mismatched_paragraphs

# Step 4: Output mismatches
def print_mismatches(mismatches):
    if mismatches:
        print("Mismatched Paragraphs:")
        for paragraph_id, xml_count, csv_count in mismatches:
            print(f"Paragraph {paragraph_id}: XML count = {xml_count}, CSV count = {csv_count}")
    else:
        print("All paragraph counts match between the XML and CSV files.")

# Paths to your files
xml_file_path = 'counted_milestones.xml'
csv_file_path = 'sentence_counts.csv'

# Execute the comparison
xml_counts = parse_xml(xml_file_path)
csv_counts = parse_csv(csv_file_path)
mismatches = compare_counts(xml_counts, csv_counts)

# Print the results
print_mismatches(mismatches)

Mismatched Paragraphs:
Paragraph : XML count = 4, CSV count = Missing in CSV
